In [21]:
import re
import math
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
print("Library berhasil diimpor.")

Library berhasil diimpor.


In [22]:
documents = [
    "Sistem komputer modern menggunakan prosesor yang sangat cepat.",
    "Jaringan komputer menghubungkan banyak perangkat komputer secara bersamaan.",
    "Kecerdasan buatan membuat komputer mampu belajar dari data.",
    "Sistem temu kembali informasi membantu pengguna mencari dokumen relevan dengan cepat.",
]

for i, d in enumerate(documents, 1):
    print(f"Dok-{i}: {d}")

Dok-1: Sistem komputer modern menggunakan prosesor yang sangat cepat.
Dok-2: Jaringan komputer menghubungkan banyak perangkat komputer secara bersamaan.
Dok-3: Kecerdasan buatan membuat komputer mampu belajar dari data.
Dok-4: Sistem temu kembali informasi membantu pengguna mencari dokumen relevan dengan cepat.


In [23]:
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()

def simple_preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = stopword_remover.remove(text)
    return text.split()


tokenized_docs = [simple_preprocess(doc) for doc in documents]

for i, toks in enumerate(tokenized_docs, 1):
    print(f"Dok-{i}: {toks}")


Dok-1: ['sistem', 'komputer', 'modern', 'menggunakan', 'prosesor', 'sangat', 'cepat']
Dok-2: ['jaringan', 'komputer', 'menghubungkan', 'banyak', 'perangkat', 'komputer', 'bersamaan']
Dok-3: ['kecerdasan', 'buatan', 'membuat', 'komputer', 'mampu', 'belajar', 'data']
Dok-4: ['sistem', 'temu', 'informasi', 'membantu', 'pengguna', 'mencari', 'dokumen', 'relevan', 'cepat']


In [24]:
vocab = sorted(set(word for toks in tokenized_docs for word in toks))

bow_rows = []
for toks in tokenized_docs:
    row = {term: toks.count(term) for term in vocab}
    bow_rows.append(row)

df_bow = pd.DataFrame(bow_rows, index=[f"Dok-{i+1}" for i in range(len(documents))])
print(df_bow)

       banyak  belajar  bersamaan  buatan  cepat  data  dokumen  informasi  jaringan  kecerdasan  komputer  mampu  \
Dok-1       0        0          0       0      1     0        0          0         0           0         1      0   
Dok-2       1        0          1       0      0     0        0          0         1           0         2      0   
Dok-3       0        1          0       1      0     1        0          0         0           1         1      1   
Dok-4       0        0          0       0      1     0        1          1         0           0         0      0   

       membantu  membuat  mencari  menggunakan  menghubungkan  modern  pengguna  perangkat  prosesor  relevan  sangat  \
Dok-1         0        0        0            1              0       1         0          0         1        0       1   
Dok-2         0        0        0            0              1       0         0          1         0        0       0   
Dok-3         0        1        0            0     

In [25]:
N = len(documents)

tf_rows = []
for toks in tokenized_docs:
    total_terms = len(toks)
    row = {term: (toks.count(term) / total_terms if total_terms > 0 else 0) for term in vocab}
    tf_rows.append(row)
df_tf = pd.DataFrame(tf_rows, index=[f"Dok-{i+1}" for i in range(N)])

print(df_tf.round(4))

       banyak  belajar  bersamaan  buatan   cepat    data  dokumen  informasi  jaringan  kecerdasan  komputer   mampu  \
Dok-1  0.0000   0.0000     0.0000  0.0000  0.1429  0.0000   0.0000     0.0000    0.0000      0.0000    0.1429  0.0000   
Dok-2  0.1429   0.0000     0.1429  0.0000  0.0000  0.0000   0.0000     0.0000    0.1429      0.0000    0.2857  0.0000   
Dok-3  0.0000   0.1429     0.0000  0.1429  0.0000  0.1429   0.0000     0.0000    0.0000      0.1429    0.1429  0.1429   
Dok-4  0.0000   0.0000     0.0000  0.0000  0.1111  0.0000   0.1111     0.1111    0.0000      0.0000    0.0000  0.0000   

       membantu  membuat  mencari  menggunakan  menghubungkan  modern  pengguna  perangkat  prosesor  relevan  sangat  \
Dok-1    0.0000   0.0000   0.0000       0.1429         0.0000  0.1429    0.0000     0.0000    0.1429   0.0000  0.1429   
Dok-2    0.0000   0.0000   0.0000       0.0000         0.1429  0.0000    0.0000     0.1429    0.0000   0.0000  0.0000   
Dok-3    0.0000   0.1429   0.00

In [26]:
df_counts = {term: sum(1 for toks in tokenized_docs if term in toks) for term in vocab}
idf_manual = {term: math.log((1 + N) / (1 + df_counts[term])) + 1 for term in vocab}

df_idf = pd.DataFrame({
    "df": [df_counts[t] for t in vocab],
    "IDF": [round(idf_manual[t], 4) for t in vocab],
}, index=vocab)

print(df_idf)

               df     IDF
banyak          1  1.9163
belajar         1  1.9163
bersamaan       1  1.9163
buatan          1  1.9163
cepat           2  1.5108
data            1  1.9163
dokumen         1  1.9163
informasi       1  1.9163
jaringan        1  1.9163
kecerdasan      1  1.9163
komputer        3  1.2231
mampu           1  1.9163
membantu        1  1.9163
membuat         1  1.9163
mencari         1  1.9163
menggunakan     1  1.9163
menghubungkan   1  1.9163
modern          1  1.9163
pengguna        1  1.9163
perangkat       1  1.9163
prosesor        1  1.9163
relevan         1  1.9163
sangat          1  1.9163
sistem          2  1.5108
temu            1  1.9163


In [27]:
tfidf_manual_rows = []
for toks in tokenized_docs:
    total_terms = len(toks)
    row = {}
    for term in vocab:
        tf_val = toks.count(term) / total_terms if total_terms > 0 else 0
        row[term] = tf_val * idf_manual[term]
    tfidf_manual_rows.append(row)

df_tfidf_manual = pd.DataFrame(tfidf_manual_rows, index=[f"Dok-{i+1}" for i in range(N)])

tfidf_manual_norm_rows = []
for row in tfidf_manual_rows:
    jumlah_kuadrat = 0
    for nilai in row.values():
        jumlah_kuadrat += nilai * nilai
    panjang = math.sqrt(jumlah_kuadrat)

    row_norm = {}
    for term, nilai in row.items():
        row_norm[term] = nilai / panjang if panjang > 0 else 0
    tfidf_manual_norm_rows.append(row_norm)

tfidf_manual_norm = pd.DataFrame(tfidf_manual_norm_rows, index=df_tfidf_manual.index)

print("TF-IDF manual (sebelum normalisasi L2):")
print(df_tfidf_manual.round(4))
print()
print("TF-IDF manual (setelah normalisasi L2):")
print(tfidf_manual_norm.round(4))


TF-IDF manual (sebelum normalisasi L2):
       banyak  belajar  bersamaan  buatan   cepat    data  dokumen  informasi  jaringan  kecerdasan  komputer   mampu  \
Dok-1  0.0000   0.0000     0.0000  0.0000  0.2158  0.0000   0.0000     0.0000    0.0000      0.0000    0.1747  0.0000   
Dok-2  0.2738   0.0000     0.2738  0.0000  0.0000  0.0000   0.0000     0.0000    0.2738      0.0000    0.3495  0.0000   
Dok-3  0.0000   0.2738     0.0000  0.2738  0.0000  0.2738   0.0000     0.0000    0.0000      0.2738    0.1747  0.2738   
Dok-4  0.0000   0.0000     0.0000  0.0000  0.1679  0.0000   0.2129     0.2129    0.0000      0.0000    0.0000  0.0000   

       membantu  membuat  mencari  menggunakan  menghubungkan  modern  pengguna  perangkat  prosesor  relevan  sangat  \
Dok-1    0.0000   0.0000   0.0000       0.2738         0.0000  0.2738    0.0000     0.0000    0.2738   0.0000  0.2738   
Dok-2    0.0000   0.0000   0.0000       0.0000         0.2738  0.0000    0.0000     0.2738    0.0000   0.0000  0

In [28]:
docs_joined = [" ".join(toks) for toks in tokenized_docs]

vectorizer = TfidfVectorizer()
tfidf_sklearn = vectorizer.fit_transform(docs_joined)

df_tfidf_sklearn = pd.DataFrame(
    tfidf_sklearn.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"Dok-{i+1}" for i in range(N)],
)[vocab]  # urutkan kolom sesuai vocab manual agar mudah dibandingkan

print(df_tfidf_sklearn.round(4))

       banyak  belajar  bersamaan  buatan   cepat    data  dokumen  informasi  jaringan  kecerdasan  komputer   mampu  \
Dok-1  0.0000   0.0000     0.0000  0.0000  0.3317  0.0000   0.0000     0.0000    0.0000      0.0000    0.2685  0.0000   
Dok-2  0.3884   0.0000     0.3884  0.0000  0.0000  0.0000   0.0000     0.0000    0.3884      0.0000    0.4958  0.0000   
Dok-3  0.0000   0.3951     0.0000  0.3951  0.0000  0.3951   0.0000     0.0000    0.0000      0.3951    0.2522  0.3951   
Dok-4  0.0000   0.0000     0.0000  0.0000  0.2746  0.0000   0.3483     0.3483    0.0000      0.0000    0.0000  0.0000   

       membantu  membuat  mencari  menggunakan  menghubungkan  modern  pengguna  perangkat  prosesor  relevan  sangat  \
Dok-1    0.0000   0.0000   0.0000       0.4207         0.0000  0.4207    0.0000     0.0000    0.4207   0.0000  0.4207   
Dok-2    0.0000   0.0000   0.0000       0.0000         0.3884  0.0000    0.0000     0.3884    0.0000   0.0000  0.0000   
Dok-3    0.0000   0.3951   0.00

In [29]:
selisih = (tfidf_manual_norm - df_tfidf_sklearn).abs()

print("Selisih absolut per term:")
print(selisih.round(6))
print(f"\nSelisih maksimum antara perhitungan manual dan scikit-learn: {selisih.values.max():.6f}")

Selisih absolut per term:
       banyak  belajar  bersamaan  buatan  cepat  data  dokumen  informasi  jaringan  kecerdasan  komputer  mampu  \
Dok-1     0.0      0.0        0.0     0.0    0.0   0.0      0.0        0.0       0.0         0.0       0.0    0.0   
Dok-2     0.0      0.0        0.0     0.0    0.0   0.0      0.0        0.0       0.0         0.0       0.0    0.0   
Dok-3     0.0      0.0        0.0     0.0    0.0   0.0      0.0        0.0       0.0         0.0       0.0    0.0   
Dok-4     0.0      0.0        0.0     0.0    0.0   0.0      0.0        0.0       0.0         0.0       0.0    0.0   

       membantu  membuat  mencari  menggunakan  menghubungkan  modern  pengguna  perangkat  prosesor  relevan  sangat  \
Dok-1       0.0      0.0      0.0          0.0            0.0     0.0       0.0        0.0       0.0      0.0     0.0   
Dok-2       0.0      0.0      0.0          0.0            0.0     0.0       0.0        0.0       0.0      0.0     0.0   
Dok-3       0.0      0.0 

In [30]:
print("Term dengan bobot TF-IDF tertinggi tiap dokumen (hasil scikit-learn):\n")
for idx in df_tfidf_sklearn.index:
    row = df_tfidf_sklearn.loc[idx]
    top_term = row.idxmax()
    print(f"{idx}: '{top_term}'  (bobot = {row[top_term]:.4f})")


Term dengan bobot TF-IDF tertinggi tiap dokumen (hasil scikit-learn):

Dok-1: 'menggunakan'  (bobot = 0.4207)
Dok-2: 'komputer'  (bobot = 0.4958)
Dok-3: 'belajar'  (bobot = 0.3951)
Dok-4: 'dokumen'  (bobot = 0.3483)


## Analisis Singkat

Pada Dokumen 1, term dengan bobot tertinggi adalah "menggunakan"/"modern"/"prosesor"
(term unik yang hanya muncul di dokumen ini), pada Dokumen 2 adalah "komputer" karena
kata ini muncul dua kali dalam dokumen tersebut sehingga TF-nya tinggi meskipun IDF-nya
lebih rendah (muncul di 3 dari 4 dokumen), pada Dokumen 3 term seperti "belajar"/"buatan"/"data"
menonjol karena unik untuk dokumen kecerdasan buatan, dan pada Dokumen 4 term seperti
"dokumen"/"informasi"/"kembali"/"mencari"/"temu" menonjol karena istilah-istilah temu
kembali informasi tersebut spesifik untuk dokumen itu saja. Secara umum, term memperoleh
bobot TF-IDF tinggi ketika ia sering muncul dalam satu dokumen tertentu (TF tinggi)
namun jarang muncul di dokumen lain (IDF tinggi) kombinasi inilah yang membuatnya
menjadi kata kunci pembeda (*discriminative term*) yang baik untuk merepresentasikan isi
dokumen tersebut dibanding kata umum seperti "komputer" yang tersebar di banyak dokumen.